In [ ]:
import os
import csv
import time
import random
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from datasets import load_dataset, Dataset
from transformers import (
    DistilBertTokenizerFast,
    DistilBertForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    TrainerCallback
)

from sklearn.metrics import f1_score
from scipy.stats import pearsonr


In [ ]:
SEED = 42
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

In [ ]:
print("\nLoading BRIGHTER dataset from Hugging Face...")

train_data = load_dataset("brighter-dataset/BRIGHTER-emotion-intensities", "eng", split="train")
val_data   = load_dataset("brighter-dataset/BRIGHTER-emotion-intensities", "eng", split="dev")
test_data  = load_dataset("brighter-dataset/BRIGHTER-emotion-intensities", "eng", split="test")

train_df = train_data.to_pandas()
val_df   = val_data.to_pandas()
test_df  = test_data.to_pandas()

print("\nOriginal split sizes:")
print("Train:", len(train_df))
print("Dev  :", len(val_df))
print("Test :", len(test_df))

print("\nSample data:")
print(train_df.head())

In [ ]:
EMOTIONS = ["anger", "fear", "joy", "sadness", "surprise"]
LEVELS = [1, 2, 3]
LABELS = [f"{emotion}_{level}" for emotion in EMOTIONS for level in LEVELS]
NUM_LABELS = len(LABELS)

print("Labels:")
print(LABELS)
print("Number of labels:", NUM_LABELS)

In [ ]:
def convert_single_step_labels(df):
    df = df.copy()

    # Remove disgust if present
    if "disgust" in df.columns:
        df = df.drop(columns=["disgust"])

    # Create 15 binary labels
    for emotion in EMOTIONS:
        df[f"{emotion}_1"] = (df[emotion] == 1).astype(int)
        df[f"{emotion}_2"] = (df[emotion] == 2).astype(int)
        df[f"{emotion}_3"] = (df[emotion] == 3).astype(int)

    return df

train_single = convert_single_step_labels(train_df)
val_single   = convert_single_step_labels(val_df)
test_single  = convert_single_step_labels(test_df)

print("\nSingle-step sample:")
print(train_single[["text"] + LABELS].head())


In [1]:
full_df = pd.concat([train_single, val_single, test_single], ignore_index=True)
full_df = full_df[["text"] + LABELS]
full_df = full_df.sample(frac=1, random_state=SEED).reset_index(drop=True)

n = len(full_df)
train_end = int(0.7 * n)
val_end = int(0.9 * n)

train_single = full_df[:train_end]
val_single   = full_df[train_end:val_end]
test_single  = full_df[val_end:]

print({
    "train": len(train_single),
    "val": len(val_single),
    "test": len(test_single)
})

# Convert pandas DataFrames to Hugging Face Dataset
train_single = Dataset.from_pandas(train_single, preserve_index=False)
val_single   = Dataset.from_pandas(val_single, preserve_index=False)
test_single  = Dataset.from_pandas(test_single, preserve_index=False)

NameError: name 'pd' is not defined

In [ ]:
tokenizer = DistilBertTokenizerFast.from_pretrained("distilbert-base-uncased")
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=128
    )

train_single = train_single.map(tokenize_function)
val_single   = val_single.map(tokenize_function)
test_single  = test_single.map(tokenize_function)

def add_labels(example):
    example["labels"] = [float(example[label]) for label in LABELS]
    return example

train_single = train_single.map(add_labels)
val_single   = val_single.map(add_labels)
test_single  = test_single.map(add_labels)

train_single.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
val_single.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
test_single.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred

    probs = 1 / (1 + np.exp(-logits))
    preds = (probs >= 0.5).astype(int)

    f1_macro = f1_score(labels, preds, average="macro", zero_division=0)
    f1_micro = f1_score(labels, preds, average="micro", zero_division=0)

    pearsons = []
    for i in range(labels.shape[1]):
        if np.std(labels[:, i]) == 0 or np.std(probs[:, i]) == 0:
            pearsons.append(0.0)
        else:
            p, _ = pearsonr(labels[:, i], probs[:, i])
            pearsons.append(0.0 if np.isnan(p) else float(p))

    pearson_mean = float(np.mean(pearsons))

    return {
        "f1_macro": f1_macro,
        "f1_micro": f1_micro,
        "pearson_mean": pearson_mean
    }



In [ ]:
from google.colab import drive
drive.mount('/content/drive')

LOG_FILE = "/content/drive/MyDrive/DistilBERT_SingleStep_log.csv"
os.makedirs("/content/drive/MyDrive", exist_ok=True)

with open(LOG_FILE, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow([
        "epoch",
        "train_loss",
        "eval_loss",
        "f1_macro",
        "f1_micro",
        "pearson_mean"
    ])

epoch_list = []
train_loss_list = []
eval_loss_list = []
f1_macro_list = []
pearson_mean_list = []

class SaveMetricsCallback(TrainerCallback):
    def __init__(self, file_path):
        self.file_path = file_path
        self.current_train_loss = None

    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs is not None and "loss" in logs and "eval_loss" not in logs:
            self.current_train_loss = float(logs["loss"])

    def on_evaluate(self, args, state, control, metrics=None, **kwargs):
        if metrics is not None:
            epoch = metrics.get("epoch", state.epoch)
            train_loss = self.current_train_loss if self.current_train_loss is not None else ""
            eval_loss = float(metrics.get("eval_loss", 0.0))
            f1_macro = float(metrics.get("eval_f1_macro", 0.0))
            f1_micro = float(metrics.get("eval_f1_micro", 0.0))
            pearson_mean = float(metrics.get("eval_pearson_mean", 0.0))

            epoch_list.append(epoch)
            train_loss_list.append(train_loss)
            eval_loss_list.append(eval_loss)
            f1_macro_list.append(f1_macro)
            pearson_mean_list.append(pearson_mean)

            with open(self.file_path, "a", newline="", encoding="utf-8") as f:
                writer = csv.writer(f)
                writer.writerow([
                    epoch,
                    train_loss,
                    eval_loss,
                    f1_macro,
                    f1_micro,
                    pearson_mean
                ])

In [ ]:
model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=NUM_LABELS,
    problem_type="multi_label_classification"
).to(device)

# TRAINING ARGUMENTS

training_args = TrainingArguments(
    output_dir="/content/distilbert_output",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=4,
    num_train_epochs=10,
    eval_strategy="epoch",
    logging_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    report_to="none",
    fp16=torch.cuda.is_available(),
    seed=SEED
)


# TRAINER

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_single,
    eval_dataset=val_single,
    data_collator=data_collator,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
    callbacks=[SaveMetricsCallback(LOG_FILE)]
)


# TRAIN

start = time.time()
trainer.train()
end = time.time()

print(f"Total training time: {end - start:.1f} seconds")
print(f"Average time per epoch: {(end - start) / training_args.num_train_epochs:.1f} seconds")
print("Epoch log file saved at:", LOG_FILE)

In [ ]:
test_results = trainer.evaluate(test_single)

print("\nTest Results:")
for key, value in test_results.items():
    print(f"{key}: {value}")


# PLOTS

plt.figure(figsize=(8, 5))
plt.plot(epoch_list, train_loss_list, marker="o")
plt.xlabel("Epoch")
plt.ylabel("Training Loss")
plt.title("Training Loss vs Epoch")
plt.grid(True)
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(epoch_list, eval_loss_list, marker="o")
plt.xlabel("Epoch")
plt.ylabel("Validation Loss")
plt.title("Validation Loss vs Epoch")
plt.grid(True)
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(epoch_list, f1_macro_list, marker="o")
plt.xlabel("Epoch")
plt.ylabel("F1 Macro")
plt.title("F1 Macro vs Epoch")
plt.grid(True)
plt.show()